# PAD-UFES-20 Class-Aware Augmentation Training

Run this notebook in Google Colab with a GPU runtime. It clones the repo, prepares PAD-UFES-20, checks DagsHub MLflow, runs the ISIC-initialized multimodal class-aware augmentation experiment, and compares the candidate against the current best metrics.

## Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Setup

In [2]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_config(name, default=None):
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            return userdata.get(name) or default
        except Exception:
            return default
    return default


def export_config(name, default=None, required=False):
    value = get_config(name, default)
    if required and not value:
        raise RuntimeError(f'Set {name} in Colab Secrets before training.')
    if value:
        os.environ[name] = str(value)
    return value


REPO_URL = 'https://github.com/SalmaneSossey/mlops-teledermatology.git'
BRANCH = 'main'
REPO_DIR = Path('/content/mlops-teledermatology')
HF_DATASET_REPO = get_config('PAD_UFES20_HF_REPO_ID', 'SalmaneExploring/pad-ufes-20')
DATA_ROOT = Path('/content/pad_ufes_20')
IMAGES_DIR = DATA_ROOT / 'all_images'
SPLITS_DIR = Path('data/processed/splits')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology')
ISIC_CHECKPOINT = DRIVE_PROJECT_DIR / 'runs/isic_2019_pretrain/efficientnet_b0_best.pt'
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'runs/multimodal_class_aware_aug/isic_init'
CANDIDATE_BUNDLE_DIR = DRIVE_PROJECT_DIR / 'model_bundles/class_aware_candidate'

RUN_TRAINING = True
BUILD_CANDIDATE_BUNDLE = False
EPOCHS = 15
BATCH_SIZE = 32
ALLOW_CPU = False
EXPERIMENT_NAME = 'pad-ufes-20-multimodal-isic-class-aware-aug'

CURRENT_BEST = {
    'test_macro_f1': 0.6902,
    'test_balanced_accuracy': 0.6804,
    'test_high_risk_recall': 0.8902,
    'SCC_recall': 0.2069,
}
TOLERANCE = 0.02

export_config('DAGSHUB_TOKEN', required=True)
export_config('DAGSHUB_USERNAME')
export_config('DAGSHUB_REPO_OWNER', 'SalmaneSossey')
export_config('DAGSHUB_REPO_NAME', 'mlops-teledermatology')
export_config('DAGSHUB_MLFLOW_TRACKING_URI')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATE_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], cwd='/content', check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())
print('Hugging Face dataset:', HF_DATASET_REPO)
print('Output dir:', OUTPUT_DIR)
print('ISIC checkpoint:', ISIC_CHECKPOINT)
print('MLflow tracking URI:', os.environ.get('DAGSHUB_MLFLOW_TRACKING_URI') or f"https://dagshub.com/{os.environ['DAGSHUB_REPO_OWNER']}/{os.environ['DAGSHUB_REPO_NAME']}.mlflow")


Working directory: /content/mlops-teledermatology
Hugging Face dataset: SalmaneExploring/pad-ufes-20
Output dir: /content/drive/MyDrive/mlops-teledermatology/runs/multimodal_class_aware_aug/isic_init
ISIC checkpoint: /content/drive/MyDrive/mlops-teledermatology/runs/isic_2019_pretrain/efficientnet_b0_best.pt
MLflow tracking URI: https://dagshub.com/SalmaneSossey/mlops-teledermatology.mlflow


## Install Dependencies

In [3]:
!pip -q install mlflow huggingface_hub scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.0/907.0 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## DagsHub And GPU Checks

In [4]:
import mlflow
import torch
from mlflow.tracking import MlflowClient

tracking_uri = os.environ.get('DAGSHUB_MLFLOW_TRACKING_URI') or f"https://dagshub.com/{os.environ['DAGSHUB_REPO_OWNER']}/{os.environ['DAGSHUB_REPO_NAME']}.mlflow"
mlflow.set_tracking_uri(tracking_uri)
experiments = MlflowClient(tracking_uri).search_experiments()
print('DagsHub token available:', bool(os.environ.get('DAGSHUB_TOKEN')))
print('MLflow tracking URI:', tracking_uri)
print('Existing experiments:', [(experiment.experiment_id, experiment.name) for experiment in experiments])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
elif RUN_TRAINING and not ALLOW_CPU:
    raise RuntimeError('Select a Colab GPU runtime before starting training.')

if not ISIC_CHECKPOINT.exists():
    raise FileNotFoundError(f'Missing ISIC checkpoint: {ISIC_CHECKPOINT}')


DagsHub token available: True
MLflow tracking URI: https://dagshub.com/SalmaneSossey/mlops-teledermatology.mlflow
Existing experiments: [('5', 'pad-ufes-20-image-baseline-multimodal-isic-init'), ('4', 'pad-ufes-20-image-baseline-isic-init'), ('3', 'pad-ufes-20-isic-2019-pretrain'), ('2', 'pad-ufes-20-image-baseline-multimodal'), ('1', 'pad-ufes-20-image-baseline-hparam-sweep'), ('0', 'pad-ufes-20-image-baseline')]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Download Data And Build Splits

In [5]:
!python -m src.data.download_pad_ufes_20 \
  --repo-id "{HF_DATASET_REPO}" \
  --output-dir "{DATA_ROOT}" \
  --force

!python -m src.data.make_image_splits \
  --metadata-path "{DATA_ROOT / 'metadata.csv'}" \
  --images-dir "{IMAGES_DIR}" \
  --output-dir "{SPLITS_DIR}"


Fetching ... files: 97it [00:30,  3.19it/s]
Fetching ... files: 1333it [04:56,  2.16it/s]
Fetching ... files: 1383it [05:11,  3.18it/s]
Fetching ... files: 1389it [05:12,  3.75it/s]
Fetching ... files: 1444it [05:23,  4.56it/s]
Fetching ... files: 1718it [06:03,  8.63it/s]
Fetching ... files: 2297it [07:14,  9.62it/s]
Fetching ... files: 2301it [07:15,  5.29it/s]
Download complete: 100% 3.57G/3.57G [07:22<00:00, 8.08MB/s]
Downloaded SalmaneExploring/pad-ufes-20 to /content/pad_ufes_20
Metadata: /content/pad_ufes_20/metadata.csv
Images: /content/pad_ufes_20/all_images
Split distribution by diagnosis:
split       test  train  val
diagnostic                  
ACK          109    511  110
BCC          127    591  127
MEL            8     36    8
NEV           36    171   37
SCC           29    134   29
SEK           35    165   35

Images per split:
split
train    1608
val       346
test      344
Name: count, dtype: int64

Patients per split:
split
train    959
val      208
test     206
Na

## Train Class-Aware Multimodal Candidate

In [6]:
if RUN_TRAINING:
    command = [
        'python', '-m', 'src.training.train_multimodal_baseline',
        '--images-dir', str(IMAGES_DIR),
        '--metadata-path', str(DATA_ROOT / 'metadata.csv'),
        '--splits-dir', str(SPLITS_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--experiment-name', EXPERIMENT_NAME,
        '--hf-dataset-repo', HF_DATASET_REPO,
        '--initial-image-checkpoint', str(ISIC_CHECKPOINT),
        '--sampler', 'weighted_random',
        '--augment-strength', 'class_aware',
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
    ]
    if ALLOW_CPU:
        command.append('--allow-cpu')
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    subprocess.run(['python', '-m', 'src.training.train_multimodal_baseline', '--help'], check=True)


Running: python -m src.training.train_multimodal_baseline --images-dir /content/pad_ufes_20/all_images --metadata-path /content/pad_ufes_20/metadata.csv --splits-dir data/processed/splits --output-dir /content/drive/MyDrive/mlops-teledermatology/runs/multimodal_class_aware_aug/isic_init --experiment-name pad-ufes-20-multimodal-isic-class-aware-aug --hf-dataset-repo SalmaneExploring/pad-ufes-20 --initial-image-checkpoint /content/drive/MyDrive/mlops-teledermatology/runs/isic_2019_pretrain/efficientnet_b0_best.pt --sampler weighted_random --augment-strength class_aware --epochs 15 --batch-size 32


If Colab runs out of memory, set `BATCH_SIZE = 16` in the setup cell and rerun from the runtime check onward.

## Compare Candidate Metrics

In [7]:
import pandas as pd

metrics_path = OUTPUT_DIR / 'multimodal_test_metrics.json'
report_path = OUTPUT_DIR / 'multimodal_classification_report.csv'
if not metrics_path.exists():
    raise FileNotFoundError(f'Missing metrics file: {metrics_path}')
if not report_path.exists():
    raise FileNotFoundError(f'Missing classification report: {report_path}')

metrics = json.loads(metrics_path.read_text())
report = pd.read_csv(report_path, index_col=0)
scc_recall = float(report.loc['SCC', 'recall'])

comparison = pd.DataFrame(
    [
        {
            'metric': 'macro F1',
            'current_best': CURRENT_BEST['test_macro_f1'],
            'candidate': float(metrics['test_macro_f1']),
            'pass_rule': float(metrics['test_macro_f1']) >= CURRENT_BEST['test_macro_f1'] - TOLERANCE,
        },
        {
            'metric': 'balanced accuracy',
            'current_best': CURRENT_BEST['test_balanced_accuracy'],
            'candidate': float(metrics['test_balanced_accuracy']),
            'pass_rule': float(metrics['test_balanced_accuracy']) >= CURRENT_BEST['test_balanced_accuracy'] - TOLERANCE,
        },
        {
            'metric': 'high-risk recall',
            'current_best': CURRENT_BEST['test_high_risk_recall'],
            'candidate': float(metrics['test_high_risk_recall']),
            'pass_rule': float(metrics['test_high_risk_recall']) >= CURRENT_BEST['test_high_risk_recall'] - TOLERANCE,
        },
        {
            'metric': 'SCC recall',
            'current_best': CURRENT_BEST['SCC_recall'],
            'candidate': scc_recall,
            'pass_rule': scc_recall > CURRENT_BEST['SCC_recall'],
        },
    ]
)
display(comparison)
display(report.loc[['BCC', 'MEL', 'SCC'], ['precision', 'recall', 'f1-score', 'support']])

if bool(comparison['pass_rule'].all()):
    print('Candidate passes the class-aware augmentation decision rule.')
else:
    print('Candidate should be reported as an ablation, not promoted as the final model.')


,metric,current_best,candidate,pass_rule
0,macro F1,0.6902,0.612654,False
1,balanced accuracy,0.6804,0.611471,False
2,high-risk recall,0.8902,0.823171,False
3,SCC recall,0.2069,0.137931,False


,precision,recall,f1-score,support
BCC,0.762295,0.732283,0.746988,127.0
MEL,0.571429,0.500000,0.533333,8.0
SCC,0.133333,0.137931,0.135593,29.0


Candidate should be reported as an ablation, not promoted as the final model.


## Optional Candidate Bundle

In [8]:
if BUILD_CANDIDATE_BUNDLE:
    command = [
        'python', '-m', 'src.inference.build_multimodal_bundle',
        '--metadata-path', str(DATA_ROOT / 'metadata.csv'),
        '--splits-dir', str(SPLITS_DIR),
        '--output-dir', str(CANDIDATE_BUNDLE_DIR),
        '--checkpoint-path', str(OUTPUT_DIR / 'efficientnet_b0_multimodal_best.pt'),
        '--mlflow-run-id', 'class-aware-colab-candidate',
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Skipping bundle build. Set BUILD_CANDIDATE_BUNDLE = True after the candidate passes review.')


Skipping bundle build. Set BUILD_CANDIDATE_BUNDLE = True after the candidate passes review.
